In [ ]:
# ==============================================================================
# RHIBS TAXONOMY INGESTION
# 
# ARCHITECTURE NOTE: The RHIBS Taxonomy is a 2-tier framework (General Category > 
# Specific Component). It is stored in a local .docx supplementary file. 
# This script uses Strategy A (Local Bulk Parsing) via the python-docx library.
#
# SSSOM ALIGNMENT STRATEGY: 
# 1. Root Nodes: The 22 General Categories serve as absolute roots.
# 2. Concept Types: Both categories and components are typed as skos:Concept.
# 3. CURIEs: Synthesized sequentially (e.g., RHIBS:1, RHIBS:1A).
# 4. Data Loss: Clinical delivery metadata (Facilitators, Evidence Base) is 
#    intentionally dropped to maintain semantic structural purity.
# ==============================================================================
# INSTRUCTIONS FOR REPRODUCIBILITY
# 1. Navigate to the published article: https://doi.org/10.1093/tbm/ibac054
# 2. Locate the "Supplementary Material" section.
# 3. Download the Word document format file (ibac054_suppl_supplementary_material.docx).
# 4. Place `ibac054_suppl_supplementary_material.docx` in your local 
#    `data/external/` directory.
# ==============================================================================

import os
import sys
import re
import pandas as pd
from docx import Document
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "RHIBS"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="Religious Health Interventions in Behavioral Sciences Taxonomy",
    fallback_uri="" # Synthetic CURIEs, no base URI
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
docx_path = os.path.join(external_data_dir, "ibac054_suppl_supplementary_material.docx")

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    print(f"[*] Overwriting previous {os.path.basename(raw_ingest_file)} due to local bulk extraction.")
    os.remove(raw_ingest_file)

# --- 4. Helper Functions ---
def clean_text(text):
    """Strips whitespace and normalizes internal line breaks."""
    if not text: return ""
    return re.sub(r'\s+', ' ', text).strip()

def parse_cell(cell):
    """
    Extracts the bolded label (up to the first line break or unbolded text) 
    and the remaining text as the description. Handles the 'NB' exception.
    """
    label_parts = []
    desc_parts = []
    label_complete = False

    for p in cell.paragraphs:
        p_text = p.text.strip()
        if not p_text: continue

        # "NB" exception: forces this paragraph and all following into the description
        if p_text.startswith("NB") or p_text.startswith("NB.") or p_text.startswith("NB:"):
            label_complete = True

        # If we move to a new paragraph and have already captured some label text,
        # the line break signifies the end of the label.
        if "".join(label_parts).strip():
            label_complete = True

        for run in p.runs:
            text = run.text
            if not text: continue
            
            # Check for soft returns (\n) or vertical tabs (\x0b) within a run
            if ('\n' in text or '\x0b' in text) and not label_complete:
                text = text.replace('\x0b', '\n')
                parts = text.split('\n', 1)
                
                if run.bold:
                    label_parts.append(parts[0])
                elif not parts[0].strip():
                    label_parts.append(parts[0])
                else:
                    label_complete = True
                    desc_parts.append(parts[0])
                
                label_complete = True
                desc_parts.append(parts[1])
                continue

            if not label_complete:
                if run.bold:
                    label_parts.append(text)
                elif not text.strip():  # Preserve intentional whitespace within the label
                    label_parts.append(text)
                else:
                    label_complete = True
                    desc_parts.append(text)
            else:
                desc_parts.append(text)
        
        # Add space at end of paragraph for description spacing
        if label_complete:
            desc_parts.append(" ")

    label = re.sub(r'\s+', ' ', "".join(label_parts)).strip()
    desc = re.sub(r'\s+', ' ', "".join(desc_parts)).strip()
    
    # Fallback if no bold text was found at all
    if not label and desc:
        label = desc
        desc = ""

    # Clean up trailing punctuation on the label (e.g., a stray colon)
    label = re.sub(r'[,:;]+$', '', label)
    
    return label, desc

def write_row(data_dict):
    clean_row = finalize_row(data_dict)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )

# --- 5. Main Extraction Loop ---
try:
    if not os.path.exists(docx_path):
        print(f"[!] Critical Error: Could not find RHIBS document at {docx_path}")
        sys.exit(1)

    print(f"[*] Loading Document: {os.path.basename(docx_path)}...")
    doc = Document(docx_path)
    
    if len(doc.tables) == 0:
        print("[!] Critical Error: No tables found in the document.")
        sys.exit(1)
        
    table = doc.tables[0]
    print(f"[*] Found primary taxonomy table with {len(table.rows)} rows.")
    
    current_cat_name = ""
    current_cat_id = 0
    spec_idx = 0
    total_processed = 0
    
    # Iterate through rows, skipping the header row
    for row in table.rows[1:]:
        cells = row.cells
        if len(cells) < 5: continue
        
        # 5a. Disentangle Labels and Descriptions via Run Parsing
        gen_label, gen_desc = parse_cell(cells[0])
        spec_label, spec_desc = parse_cell(cells[1])
        examples = clean_text(cells[2].text)
        
        # Skip completely empty rows
        if not gen_label and not spec_label: continue
        
        # 5b. Detect and Process New General Category
        # Because python-docx repeats merged cell text across rows, a change 
        # in gen_label signals a new category boundary
        if gen_label and gen_label != current_cat_name:
            current_cat_id += 1
            current_cat_name = gen_label
            spec_idx = 0
            
            # Formulate Description
            cat_desc = gen_desc
            if examples: cat_desc += f" Examples: {examples}"
            
            write_row({
                'Source_System': SOURCE_NAME,
                'Primary_Label': current_cat_name,
                'CURIE': f"{SOURCE_PREFIX}:{current_cat_id}",
                'Formal_Label': "",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': current_cat_name,
                'Description': cat_desc.strip(),
                'Parent_IDs': "",
                'Concept_ID': str(current_cat_id),
                'Status': "active",
            })
            total_processed += 1
            print(f"\r[Ingesting] Root: {current_cat_name[:40]:<40} (ID: {current_cat_id})", end="", flush=True)
            
        # 5c. Process Specific Component
        if spec_label:
            spec_char = chr(65 + spec_idx) # Converts 0 -> A, 1 -> B, etc.
            spec_id_str = f"{current_cat_id}{spec_char}"
            
            # Formulate Description
            child_desc = spec_desc
            if examples: child_desc += f" Examples: {examples}"
            
            write_row({
                'Source_System': SOURCE_NAME,
                'Primary_Label': spec_label,
                'CURIE': f"{SOURCE_PREFIX}:{spec_id_str}",
                'Formal_Label': "",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': f"{current_cat_name} > {spec_label}",
                'Description': child_desc.strip(),
                'Parent_IDs': str(current_cat_id),
                'Concept_ID': spec_id_str,
                'Status': "active",
            })
            total_processed += 1
            spec_idx += 1
            
except Exception as e:
    print(f"\n[!] Error during parsing: {e}")

print(f"\n\n[COMPLETE] Extracted {total_processed} RHIBS concepts to Bronze Layer.")